# Projeto 5: Classificação binária brest cancer - carregar o classificador

## Etapa 1: Importação das bibliotecas

ambiente conda activate YouTube, no linux

In [36]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score
torch.__version__

'2.8.0+cu128'

## Etapa 2: Carregamento do classificador

Estamos fazendo a classificação binária, salvamos os dados como 'classificador.pth', usando o comando torch.save(classificador.state_dict, 'classificador.pth')

agora aprendemos como carregar o classificador. Fazendo uma previsão. 

In [37]:
np.random.seed(123) # fixar sempre os mesmos valores aleátórios de geração de numpy. 
torch.manual_seed(123) # inicialização da semente aleatória, fixando sempre o mesmo falor 

In [38]:
# necessário redefinir toda a estrutura da rede neural
#Cria uma classe que herda de nn.Module, ou seja, um modelo treinável em PyTorch.
class classificador_torch(nn.Module):
    def __init__(self):
        #Inicializa a classe base (nn.Module).
        super().__init__()
        #Primeira camada totalmente conectada (linear).
        #Recebe 30 entradas (ex: 30 variáveis preditoras) e gera 8 neurônios.      
        #Os pesos são inicializados com uma distribuição normal (N(0, 0.05)).
        self.dense0 = nn.Linear(30, 16) # torquei o 8 por 16 poos nas aulas anteriores tiveram os maiores desempenhos
        torch.nn.init.normal_(self.dense0.weight, mean = 0.0, std = 0.05)
        #Segunda camada linear.        
        #Mantém 8 neurônios → 8 neurônios.        
        #Também inicializada com distribuição normal.        
        self.dense1 = nn.Linear(16, 16)
        torch.nn.init.normal_(self.dense1.weight, mean = 0.0, std = 0.05)
        #Camada de saída.       
        #Reduz para 1 neurônio (típico de classificação binária).        
        self.dense2 = nn.Linear(16, 1)
        #ReLU: função de ativação nas camadas ocultas.        
        #Dropout(0.2): desliga aleatoriamente 20% dos neurônios durante o treino → reduz overfitting.        
        #Sigmoid: saída entre 0 e 1, ideal para classificação binária (probabilidade da classe positiva).        
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.output = nn.Sigmoid()

    #Fluxo de dados:    
    #Entrada → dense0 → ReLU → Dropout    
    #Passa para dense1 → ReLU → Dropout    
    #Vai para dense2 → Sigmoid (probabilidade final)    
    def forward(self, X):
        X = self.dense0(X)
        X = self.activation(X)
        X = self.dropout(X)
        X = self.dense1(X)
        X = self.activation(X)
        X = self.dropout(X)
        X = self.dense2(X)
        X = self.output(X)
        return X

$\begin{array}{|c|c|c|c|}
\hline
\textbf{Função} & \textbf{Fórmula} & \textbf{Intervalo de saída} & \textbf{Uso comum} \\
\hline
\text{Sigmoid} & 
\sigma(x) = \frac{1}{1 + e^{-x}} & (0, 1) & \text{Classificação binária} \\
\hline
\text{Tanh} & 
\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}} & (-1, 1) & \text{Camadas ocultas (alternativa ao ReLU)} \\
\hline
\text{ReLU} & 
f(x) = \max(0, x) & [0, \infty) & \text{Camadas ocultas (mais usado)} \\
\hline
\text{Leaky ReLU} & 
f(x) = \begin{cases}
x & \text{se } x > 0 \\
\alpha x & \text{se } x \leq 0
\end{cases} & (-\infty, \infty) & \text{Evita neurônios mortos no ReLU} \\
\hline
\text{ELU} & 
f(x) = \begin{cases}
x & \text{se } x > 0 \\
\alpha(e^x - 1) & \text{se } x \leq 0
\end{cases} & (-\alpha, \infty) & \text{Camadas ocultas, alternativa ao ReLU} \\
\hline
\text{Softmax} & 
f(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}} & (0, 1), \ \sum f(x_i) = 1 & \text{Classificação multiclasse} \\
\hline
\end{array}$

# 🔹 O ReLU (Rectified Linear Unit)

Características:

Simples, rápido de calcular.

Resolve o problema do vanishing gradient (quando os gradientes ficam quase zero em sigmoides/tanh).

Muito usado em camadas ocultas.

⚠️ Problema: pode causar “neurônios mortos” (quando muitos valores ficam ≤ 0 e nunca mais reativam).


# 🔹 1. Sigmoid

Saída entre 0 e 1 → ideal para classificação binária.

Problema: saturação → gradientes muito pequenos para valores extremos.

# 🔹 2. Tanh (Tangente Hiperbólica)

Saída entre -1 e 1.

Centraliza os dados em torno de 0 (melhor que sigmoid).

Mas ainda sofre com vanishing gradient.

# 🔹 3. Leaky ReLU

Variante do ReLU que deixa um “vazamento” para valores negativos.

Resolve o problema de neurônios mortos.

#🔹 4. ELU (Exponential Linear Unit)

Parecido com ReLU, mas valores negativos seguem uma curva exponencial suave.
Melhora a aprendizagem, mas é mais caro computacionalmente.

#🔹 5. Softmax

Transforma um vetor de valores em probabilidades que somam 1.

Usado em classificação multiclasse (saída com várias categorias).



🔹 Resumo, o uso mais comum:

Camadas ocultas → ReLU, Leaky ReLU, às vezes Tanh/ELU.

Saída binária → Sigmoid.

Saída multiclasse → Softmax.

### . Nas camadas ocultas

ReLU → é o mais comum, simples e eficiente, funciona bem na prática nas séries temporais.

Leaky ReLU / ELU → boas alternativas se você notar neurônios mortos ou quiser capturar melhor valores negativos.

Tanh → ainda é bastante usado em RNNs, LSTMs, GRUs, porque ajuda a manter os valores centralizados em torno de 0, o que estabiliza a recorrência.

#### 🔹 2. Na camada de saída

Depende do tipo de previsão que se faz:

Previsão de valores contínuos (regressão) → sem ativação (linear) na saída.
Exemplo: prever vendas, temperatura, preço → queremos números reais, não limitados.

Classificação binária de séries temporais (ex: prever se vai ter queda ou não) → Sigmoid.

Classificação multiclasse temporal (ex: prever estado do mercado: queda, estável, alta) → Softmax.

🔹 3. Resumindo

Camadas ocultas (feature extraction):
ReLU / Leaky ReLU → se for rede feedforward ou CNNs.
Tanh / Sigmoid → se for RNN/LSTM/GRU.

Camada de saída (target):

Regressão (valor contínuo): Linear (sem ativação).

Classificação binária: Sigmoid.

Classificação multiclasse: Softmax.

### valores numéricos de séries temporais (ex: vendas, demanda, preços), o ideal é:

- ReLU nas camadas ocultas

- Linear (sem ativação) na saída.

#--------------------------------------------------------

### pytorch não tem uma função para salvar a estrutura da rede neural. 

### por isso definimos a rede neural da super em diante, codificamos como classe que segue acima

In [39]:
# inicializar o classificador

classificador = classificador_torch()

# agora vamos carregar o que salvamos em .pth
state_dict = torch.load('checkpoint.pth') # ideal sempre com esse nome checkpoint

In [40]:
# foi salvo a estrutura da rede neural
state_dict

OrderedDict([('dense0.weight',
              tensor([[ 3.0197e-02,  1.8140e-01,  1.6734e-01, -3.2692e-04, -1.6838e-01,
                        4.5863e-04, -2.0909e-01,  6.0946e-02,  2.3099e-01,  2.0604e-01,
                        4.8386e-02, -9.7833e-02, -1.8874e-02, -7.5993e-02, -5.5547e-03,
                       -5.2949e-02,  8.0095e-02,  2.0435e-01, -3.8628e-03, -1.0071e-02,
                        2.1744e-02,  1.6817e-01,  2.7692e-02, -3.8436e-02, -8.1606e-02,
                        2.3104e-02, -4.2568e-02, -7.5688e-02,  1.3744e-01,  4.5682e-04],
                      [-4.9082e-02, -1.1097e-01, -2.0753e-01, -6.7641e-02,  1.6660e-01,
                        9.6760e-02,  9.3890e-02,  2.0316e-02, -6.9786e-02, -1.1045e-01,
                        3.5867e-03,  8.0619e-04,  6.4567e-03, -1.8769e-02,  1.1516e-02,
                        2.6228e-02, -4.3028e-02,  7.3910e-02,  5.5967e-02,  1.8892e-01,
                        1.0440e-02, -2.2173e-02, -1.9161e-01,  1.0880e-01, -4.7178e-02,


In [41]:
classificador # estrutura da rede neural pronta

classificador_torch(
  (dense0): Linear(in_features=30, out_features=16, bias=True)
  (dense1): Linear(in_features=16, out_features=16, bias=True)
  (dense2): Linear(in_features=16, out_features=1, bias=True)
  (activation): ReLU()
  (dropout): Dropout(p=0.2, inplace=False)
  (output): Sigmoid()
)

In [42]:
# meu simulador aula M3-4

#classificador_torch(
#  (dense0): Linear(in_features=30, out_features=16, bias=True)
#  (dense1): Linear(in_features=16, out_features=16, bias=True)
#  (dense2): Linear(in_features=16, out_features=1, bias=True)
#  (activation): ReLU()
#  (dropout): Dropout(p=0.2, inplace=False)
#  (output): Sigmoid()
#)


# o da aula 

#classificador_torch(
#  (dense0): Linear(in_features=30, out_features=8, bias=True)
#  (dense1): Linear(in_features=8, out_features=8, bias=True)
#  (dense2): Linear(in_features=8, out_features=1, bias=True)
#  (activation): ReLU()
#  (dropout): Dropout(p=0.2, inplace=False)
#  (output): Sigmoid()
#)
#por isso o processo não rodou , números diferentes dos neurônios

#classificador.load_state_dict(state_dict) # todas as caixas se encaixaram perfeitamente, canais, neurônios

# carregar tudo que está na variável

In [43]:
classificador.load_state_dict(state_dict) # todas as caixas se encaixaram perfeitamente, canais, neurônios

<All keys matched successfully>

## Etapa 3: Previsões


In [45]:
# colocamos os novos valores das colunas de previsores

#(Index([' radius_mean', ' texture_mean', ' perimeter_mean', ' area_mean',
#        ' smoothness_mean', ' compactness_mean', ' concavity_mean',
#        'concave_points_mean', ' symmetry_mean', ' fractal_dimension_mean',
#        ' radius_se', ' texture_se', ' perimeter_se', ' area_se',
#        ' smoothness_se', ' compactness_se', ' concavity_se',
#        ' concave_points_se', ' symmetry_se', ' fractal_dimension_se',
#        ' radius_worst', ' texture_worst', ' perimeter_worst', ' area_worst',
#        ' smoothness_worst', ' compactness_worst', ' concavity_worst',
#        ' concave_points_worst', ' symmetry_worst', ' fractal_dimension_worst'],
#       dtype='object'),
# Index(['0'], dtype='object'))

In [46]:
novo = torch.tensor([[15.80, 8.34, 118, 900, 0.10, 0.26, 0.08, 0.134, 0.178,
                      0.20, 0.05, 1098, 0.87, 4500, 145.2, 0.005, 0.04, 0.05, 0.015,
                      0.03, 0.007, 23.15, 16.64, 178.5, 2018, 0.14, 0.185,
                      0.84, 158, 0.363]])

In [49]:
classificador.eval()
previsao = classificador(novo)
previsao = (previsao.detach().numpy() > 0.5) # colocar ou selecionar a probabilidade maior que 0,5, para filtrar os dados mais acertivos 
previsao # retornado assim true ou falso, no exemplo ficou true, o que equivale um câncer maligno

array([[ True]])

# outra previsão, com base nos dados carregados.

In [50]:
previsores = pd.read_csv('entradas_breast.csv')
classe = pd.read_csv('saidas_breast.csv')

In [52]:
previsores.tail(10)

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,symmetry_mean,fractal_dimension_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
559,11.51,23.93,74.52,403.5,0.09261,0.10210,0.11120,0.04105,0.1388,0.06570,...,12.48,37.16,82.28,474.2,0.12980,0.25170,363.0000,0.09653,0.2112,0.08732
560,14.05,27.15,91.38,600.4,0.09929,0.11260,0.04462,0.04304,0.1537,0.06171,...,15.30,33.17,100.20,706.7,0.12410,0.22640,0.1326,0.10480,225.0000,0.08321
561,11.20,29.37,70.67,386.0,0.07449,0.03558,0.00000,0.00000,106.0000,0.05502,...,11.92,38.30,75.19,439.6,0.09267,0.05494,0.0000,0.00000,0.1566,0.05905
562,15.22,30.62,103.40,716.9,0.10480,0.20870,255.00000,0.09429,0.2128,0.07152,...,17.52,42.79,128.70,915.0,0.14170,0.79170,1.1700,0.23560,0.4089,0.14090
563,20.92,25.09,143.00,1347.0,0.10990,0.22360,0.31740,0.14740,0.2149,0.06879,...,24.29,29.41,179.10,1819.0,0.14070,0.41860,0.6599,0.25420,0.2929,0.09873
564,21.56,22.39,142.00,1479.0,111.00000,0.11590,0.24390,0.13890,0.1726,0.05623,...,25.45,26.40,166.10,2027.0,141.00000,0.21130,0.4107,0.22160,206.0000,0.07115
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,144.00000,0.09791,0.1752,0.05533,...,23.69,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.16280,0.2572,0.06637
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,159.0000,0.05648,...,18.98,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.14180,0.2218,0.07820
567,20.60,29.33,140.10,1265.0,0.11780,277.00000,0.35140,152.00000,0.2397,0.07016,...,25.74,39.42,184.60,1821.0,165.00000,0.86810,0.9387,265.00000,0.4087,124.00000
568,7.76,24.54,47.92,181.0,0.05263,0.04362,0.00000,0.00000,0.1587,0.05884,...,9456.00,30.37,59.16,268.6,0.08996,0.06444,0.0000,0.00000,0.2871,0.07039


In [53]:
classe.tail()

,0
564,0
565,0
566,0
567,0
568,1


## formatar para o pytorch, pois precisa transformar em array e depois para tensor

In [54]:
previsores = torch.tensor(np.array(previsores), dtype = torch.float)
classe = torch.tensor(np.array(classe), dtype = torch.float)

In [55]:
previsores

tensor([[1.7990e+01, 1.0380e+01, 1.2280e+02,  ..., 2.6540e-01, 4.6010e-01,
         1.1890e-01],
        [2.0570e+01, 1.7770e+01, 1.3290e+02,  ..., 1.8600e+02, 2.7500e+02,
         8.9020e-02],
        [1.9690e+01, 2.1250e+01, 1.3000e+02,  ..., 2.4300e+02, 3.6130e-01,
         8.7580e-02],
        ...,
        [1.6600e+01, 2.8080e+01, 1.0830e+02,  ..., 1.4180e-01, 2.2180e-01,
         7.8200e-02],
        [2.0600e+01, 2.9330e+01, 1.4010e+02,  ..., 2.6500e+02, 4.0870e-01,
         1.2400e+02],
        [7.7600e+00, 2.4540e+01, 4.7920e+01,  ..., 0.0000e+00, 2.8710e-01,
         7.0390e-02]])

In [58]:
classe[:12]

tensor([[0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.]])

previsores que vai dar as respostas 

In [59]:
# vai passar os dados para a rede neural (foward)
previsoes = classificador.forward(previsores).detach()

detach, que precisa retirar os atributos no final que acuma as operações feitas no tenso, que é usado no cálculo do gradiente. colocado automáticamente no pytorch. 

In [61]:
# previsoes são os valores das probabilidades, com valores de 0 e 1.

previsoes

tensor([[7.6688e-09],
        [5.8071e-06],
        [2.1087e-04],
        [7.8749e-05],
        [3.3044e-03],
        [2.0006e-01],
        [1.6647e-06],
        [3.6686e-02],
        [5.4888e-02],
        [4.7558e-10],
        [1.5421e-02],
        [4.0165e-04],
        [9.6144e-02],
        [8.0195e-01],
        [8.8344e-01],
        [1.1334e-01],
        [3.0499e-03],
        [1.5940e-04],
        [4.8946e-11],
        [9.9103e-01],
        [8.1379e-01],
        [1.0000e+00],
        [2.1669e-01],
        [1.8547e-11],
        [8.9095e-12],
        [5.9898e-05],
        [1.0834e-01],
        [3.2340e-03],
        [9.8263e-06],
        [7.9937e-02],
        [7.2826e-05],
        [4.3610e-03],
        [1.1461e-03],
        [1.2405e-06],
        [5.2859e-04],
        [1.9003e-02],
        [6.6556e-01],
        [9.9652e-01],
        [3.6534e-01],
        [2.7324e-02],
        [3.2833e-01],
        [9.1421e-01],
        [5.2040e-05],
        [2.9980e-02],
        [6.1298e-01],
        [1

In [62]:
# Fazer o cálculo 
# previsoes probabilidade de 0 e 1, e o arquivo classe
# importa o functional, para ter o retorno do valor do erro na base dos dados
# import torch.nn.functional as F
F.binary_cross_entropy(previsoes, classe).numpy()

array(0.13270709, dtype=float32)

In [63]:
# erro de 13% no exemplo

In [64]:

# classe como numpy, com acurácia boa, que é na base de dados do treinamento a classe, e o arquivo previsoes é a previsão do lstm+pytorch
accuracy_score(classe.numpy(), (previsoes > 0.5).numpy())

0.9437609841827768

In [73]:

# classe como numpy, com acurácia boa, que é na base de dados do treinamento a classe, e o arquivo previsoes é a previsão do lstm+pytorch
print('acurácia de {:.2f} %.'.format(accuracy_score(classe.numpy(), (previsoes > 0.5).numpy())*100))

acurácia de 94.38 %.


# Binary Cross-Entropy (BCE)

A função **Binary Cross-Entropy (BCE)** mede o erro entre as previsões de probabilidade e as classes reais (0 ou 1).  
É usada em problemas de **classificação binária**.

---

## Fórmula

$$
\text{BCE}(y, \hat{y}) = - \frac{1}{N} \sum_{i=1}^N \Big[ y_i \cdot \log(\hat{y}_i) + (1-y_i)\cdot \log(1-\hat{y}_i) \Big]
$$

Onde:

- $\( y_i \)$ = valor real (0 ou 1)  
- $\( \hat{y}_i \)$ = previsão do modelo (probabilidade entre 0 e 1)  

---

## Exemplo em PyTorch

In [74]:


#```python
import torch
import torch.nn.functional as F

# Previsões do modelo (após sigmoid)
previsoes = torch.tensor([0.9, 0.2, 0.8, 0.4], dtype=torch.float32)

# Classes reais
classe = torch.tensor([1, 0, 1, 0], dtype=torch.float32)

# Cálculo do erro BCE
erro = F.binary_cross_entropy(previsoes, classe)

print("Erro BCE:", erro.item())

Erro BCE: 0.26561832427978516


# Exemplo de cálculo manual da Binary Cross-Entropy (BCE)

---

## Dados

Previsões:
$\[
\hat{y} = [0.9, \; 0.2]
\]$

Classes reais:
$\[
y = [1, \; 0]
\]$

---

## Fórmula geral

$$
\text{BCE}(y, \hat{y}) = - \frac{1}{N} \sum_{i=1}^N \Big[ y_i \cdot \log(\hat{y}_i) + (1-y_i)\cdot \log(1-\hat{y}_i) \Big]
$$

---

## Substituindo os valores

Com \( N = 2 \):

$$
\text{BCE} = -\frac{1}{2} \Big[ 1 \cdot \log(0.9) + (1 - 0)\cdot \log(1 - 0.2) \Big]
$$

---

## Simplificando

$$
\text{BCE} = -\frac{1}{2} \Big[ \log(0.9) + \log(0.8) \Big]
$$

---

## Valores numéricos

- $\(\log(0.9) \approx -0.1053\)$  
- $\(\log(0.8) \approx -0.2231\)$  

Portanto:

$$
\text{BCE} = -\frac{1}{2} \Big[ -0.1053 + -0.2231 \Big]
$$

$$
\text{BCE} = -\frac{1}{2} \Big[ -0.3284 \Big]
$$

$$
\text{BCE} = 0.1642
$$

---

✅ Resultado final:

$\[
\text{BCE} \approx 0.1642
\]$

In [75]:
#```python
import torch
import torch.nn.functional as F

# Previsões do modelo (após sigmoid)
previsoes = torch.tensor([0.9, 0.2], dtype=torch.float32)

# Classes reais
classe = torch.tensor([1, 0], dtype=torch.float32)

# Cálculo do erro BCE
erro = F.binary_cross_entropy(previsoes, classe)

print("Erro BCE:", erro.item())

Erro BCE: 0.16425205767154694
